# 1. Read data & visualize

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

def resolve_existing_path(*candidates):
    for candidate in candidates:
        path = Path(candidate).resolve()
        if path.exists():
            return path
    raise FileNotFoundError(f"None of these paths exist: {candidates}")

BASE_PATH = resolve_existing_path("../data/dogncat", "data/dogncat")
TRAIN_PATH = BASE_PATH / "train"
VAL_PATH = BASE_PATH / "val"
TEST_PATH = BASE_PATH / "test"
SAMPLE_SUBMISSION_PATH = BASE_PATH / "sampleSubmission.csv"

MODELS_PATH = resolve_existing_path("../outputs/dogncat", "outputs/dogncat")
BEST_MODEL_PATH = MODELS_PATH / "efficientnet_b2 (1).pth"

for path in [TRAIN_PATH, VAL_PATH, TEST_PATH, SAMPLE_SUBMISSION_PATH, BEST_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Data path: {BASE_PATH}")
print(f"Models path: {MODELS_PATH}")

In [ ]:
train = sorted(str(path.relative_to(TRAIN_PATH)) for path in TRAIN_PATH.rglob("*.jpg"))
print(type(train))
print(len(train))
print(train[:5])

In [ ]:
test = sorted((path.name for path in TEST_PATH.glob("*.jpg")), key=lambda name: int(Path(name).stem))
print(len(test))
print(test[:5])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plt.figure(figsize = (10,10))
for i in range(9):
    plt.subplot(3,3,i + 1)
    path = TRAIN_PATH / train[i]
    img = mpimg.imread(path)
    print(img.shape)
    plt.imshow(img)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cat = 0
for img_path in train:
    if "cat" in img_path:
        cat += 1
dog = len(train) - cat

print("Target distribution")
print(f"Cat: {cat * 100 / len(train)}%")
print(f"Dog: {dog * 100 / len(train)}%")

In [ ]:
train_folders = sorted(path.name for path in TRAIN_PATH.iterdir() if path.is_dir())
print("Train folders:", train_folders)
print("Data is already organized for ImageFolder; skip moving files.")

In [ ]:
train_folder = sorted(path.name for path in TRAIN_PATH.iterdir() if path.is_dir())
print(train_folder)

In [ ]:
cat_folder = [path for path in TRAIN_PATH.rglob("*.jpg") if path.name.startswith("cat.")]
dog_folder = [path for path in TRAIN_PATH.rglob("*.jpg") if path.name.startswith("dog.")]

print(f"Number of cat's image: {len(cat_folder)}")
print(f"Number of dog's image: {len(dog_folder)}")

# 2. Prepare dataset

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale = (0.7, 1.0)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.RandomRotation(degrees = 15),
    transforms.ColorJitter(
        brightness = 0.3, saturation = 0.2,
        contrast = 0.3, hue = 0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder(
    root = TRAIN_PATH,
    transform = train_transform
)

print("Class to index mapping:", train_dataset.class_to_idx)
print("Classes:", train_dataset.classes)

img, label = train_dataset[0]
print(type(img))
print("Label index:", label)
print("Label name:", train_dataset.classes[label])

dataloader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
imgs, labels = next(iter(dataloader))

fig, axes = plt.subplots(2,4, figsize = (12, 6))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # de-normalize
    ax.imshow(img.clip(0, 1))
    ax.axis('off')
plt.show()

In [ ]:
train_dataset = datasets.ImageFolder(TRAIN_PATH, transform = train_transform)
val_dataset = datasets.ImageFolder(VAL_PATH, transform = val_transform)

train_size = len(train_dataset)
val_size = len(val_dataset)

train_set = train_dataset
val_set = val_dataset

print("Train classes:", train_dataset.class_to_idx)
print("Val classes:", val_dataset.class_to_idx)
print(f"Train size: {train_size} | Val size: {val_size}")

In [ ]:
# train_labels = torch.tensor([label for _, label in train_set])
# val_labels = torch.tensor([label for _, label in val_set])

# cat_train = (train_labels == 0).sum().item()
# dog_train = (train_labels == 1).sum().item()

# cat_val = (val_labels == 0).sum().item()
# dog_val = (val_labels == 1).sum().item()

# print("Distribution after split")
# print(f"Train: Cat = {cat_train * 100 / train_size}%  |  Dog = {dog_train * 100 / train_size}%")
# print(f"Val: Cat = {cat_val * 100 / val_size}%  |  Dog = {dog_val * 100 / val_size}%")

Distribution after split
- Train: Cat = 49.96%  |  Dog = 50.04%
- Val: Cat = 50.36%  |  Dog = 49.64%

In [ ]:
train_dataloader = DataLoader(train_set, batch_size = 64, shuffle = True)
val_dataloader = DataLoader(val_set, batch_size = 64)

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class TestDataset(Dataset):
    def __init__(self, root, transform = None):
        image_dirs = []
        self.transform = transform
        self.root = root
        
        try:
            image_dirs = os.listdir(root)
        except Exception as e:
            raise e

        self.image_dirs = sorted(image_dirs, key = lambda x: int(x.split(".")[0]))
        print(f"First 5 image paths: {self.image_dirs[:5]}")

    def __len__(self):
        return len(self.image_dirs)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.image_dirs[idx])
        
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            print(f"Error at image {idx}, skip to the next imamge...")
            next_idx = (idx + 1) % len(self.image_dirs)
            image = self.__getitem__(next_idx)

        if self.transform:
            image = self.transform(image)

        return image

test_dataset = TestDataset(root = TEST_PATH, transform = val_transform)

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size = 8)
imgs = next(iter(test_dataloader))

fig, axes = plt.subplots(2,4, figsize = (12, 6))
for i, ax in enumerate(axes.flat):
    img = imgs[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # de-normalize
    ax.imshow(img.clip(0, 1))
    ax.axis('off')
plt.show()

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size = 64)

In [ ]:
full_train_dataloader = DataLoader(train_dataset, batch_size = 64)

# 3. Train & Evaluate

In [ ]:
import torch.nn as nn
import torchvision.models as models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
os.makedirs(MODELS_PATH, exist_ok = True)
print("Done")

In [ ]:
import pandas as pd

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    train_loss = 0.0
    model.train()
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)

        train_loss += loss.item()
        loss.backward()
        optimizer.step()
    return train_loss / len(dataloader)
    

def validate(model, dataloader, criterion, device):
    val_loss = 0.0
    model.eval()

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    return val_loss / len(dataloader)
    

def submit(model, dataloader, device, submission_name = None):
    sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    probs = []

    model.eval()
    with torch.no_grad():
        for images in dataloader:
            images = images.to(device)
            logits = model(images)
            probs.extend(torch.sigmoid(logits).squeeze(1).cpu().numpy())

    print("Submit successfully.")
    sub["label"] = probs
    if submission_name:
        sub_name = submission_name + "_submission.csv"
    else:
        sub_name = "submission.csv"
    sub.to_csv(sub_name, index = False)
    return sub            

## 3.1. MobileNet v3 small

In [ ]:
# mobilenet_v3_small = models.mobilenet_v3_small(weights = models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)

In [ ]:
# last_layer = mobilenet_v3_small.classifier[-1]
# in_features = last_layer.in_features
# mobilenet_v3_small.classifier[-1] = nn.Linear(in_features, 1)
# mobilenet_v3_small.classifier

In [ ]:
# mobilenet_v3_small_gpu = mobilenet_v3_small.to(device)

# for param in mobilenet_v3_small_gpu.parameters():
#      param.requires_grad = False
# for param in mobilenet_v3_small_gpu.classifier.parameters():
#      param.requires_grad = True

# criterion = nn.BCEWithLogitsLoss()
# optimizer_stage1 = torch.optim.Adam(mobilenet_v3_small_gpu.classifier.parameters(), lr = 1e-3)

In [ ]:
# best_model_path = os.path.join(MODELS_PATH, "mobilenet_v3_small.pth")

# best_val_loss = float("inf")

# print("Stage 1...")
# EPOCHS = 5
# train_s1 = []
# val_s1 = []

# for e in range(EPOCHS):
#     train_loss = train_one_epoch(mobilenet_v3_small_gpu, train_dataloader, optimizer_stage1, criterion, device)
#     val_loss = validate(mobilenet_v3_small_gpu, val_dataloader, criterion, device)

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(mobilenet_v3_small_gpu.state_dict(), best_model_path)

#     print(f"EPOCHS [{e + 1}/{EPOCHS}] : train = {train_loss} | val = {val_loss}")
#     train_s1.append(train_loss)
#     val_s1.append(val_loss)

In [ ]:
# mobilenet_v3_small_gpu.load_state_dict(torch.load(best_model_path))
# for param in mobilenet_v3_small_gpu.parameters():
#     param.requires_grad = True
# optimizer_stage2 = torch.optim.Adam(mobilenet_v3_small_gpu.parameters(), lr = 1e-5)

# print("Stage 2...")
# EPOCHS_S2 = 10
# train_s2 = []
# val_s2 = []

# for e in range(EPOCHS_S2):
#     train_loss = train_one_epoch(mobilenet_v3_small_gpu, train_dataloader, optimizer_stage2, criterion, device)
#     val_loss = validate(mobilenet_v3_small_gpu, val_dataloader, criterion, device)

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         torch.save(mobilenet_v3_small_gpu.state_dict(), best_model_path)

#     print(f"EPOCHS [{e + 1}/{EPOCHS_S2}] : train = {train_loss} | val = {val_loss}")
#     train_s2.append(train_loss)
#     val_s2.append(val_loss)

In [ ]:
# mobilenet_v3_small_gpu.load_state_dict(torch.load(best_model_path))
# for param in mobilenet_v3_small_gpu.parameters():
#     param.requires_grad = True
# optimizer_stage3 = torch.optim.Adam(mobilenet_v3_small_gpu.parameters(), lr = 1e-5)

# print("Stage 3...")
# EPOCHS_S3 = 5

# for e in range(EPOCHS_S3):
#     train_loss = train_one_epoch(mobilenet_v3_small_gpu, full_train_dataloader, optimizer_stage3, criterion, device)
#     print(f"EPOCHS [{e + 1}/{EPOCHS_S3}] : train = {train_loss}")

In [ ]:
# mobilenet_v3_small_gpu.load_state_dict(torch.load("/kaggle/input/models/daoviet/mobilenet-v3-small/pytorch/default/1/mobilenet_v3_small.pth"))
# print("Load model successfully.")

# sub = submit(mobilenet_v3_small_gpu, test_dataloader, device)

## 3.2. EfficientNet b2

In [ ]:
efficientnet_b2 = models.efficientnet_b2(weights = models.EfficientNet_B2_Weights)
print("Load pretrained model successfully.")

In [ ]:
last_layer = efficientnet_b2.classifier[-1]
in_features = last_layer.in_features
efficientnet_b2.classifier[-1] = nn.Linear(in_features, 1)
print(efficientnet_b2.classifier[-1])

In [ ]:
for param in efficientnet_b2.parameters():
    param.requires_grad = False
for param in efficientnet_b2.classifier.parameters():
    param.requires_grad = True

efficientnet_b2_gpu = efficientnet_b2.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer_1 = torch.optim.Adam(efficientnet_b2_gpu.classifier.parameters(), lr = 1e-5)
print("Done setup stage 1.")

In [ ]:
print("Stage 1...")

model_path = os.path.join(MODELS_PATH, "efficientnet_b2.pth")
train_loss_s1 = []
val_loss_s1 = []
EPOCHS_S1 = 3
best_val_loss = float("inf")

# for e in range(EPOCHS_S1):
#    train = train_one_epoch(efficientnet_b2_gpu, train_dataloader, optimizer_1, criterion, device)
#    val = validate(efficientnet_b2_gpu, val_dataloader, criterion, device)

#    if val < best_val_loss:
#        best_val_loss = val
#        torch.save(efficientnet_b2_gpu.state_dict(), model_path)

#    print(f"EPOCH [{e + 1}/{EPOCHS_S1}] : train = {train} | val = {val}")
#    train_loss_s1.append(train)
#    val_loss_s1.append(val)

In [ ]:
print("Stage 2...")

best_model_path = BEST_MODEL_PATH

efficientnet_b2_gpu.load_state_dict(torch.load(best_model_path))
print("Load best model successfully.")

for param in efficientnet_b2_gpu.parameters():
    param.requires_grad = True

optimizer_2 = torch.optim.Adam(efficientnet_b2_gpu.parameters(), lr = 1e-5)
train_loss_s2 = []
val_loss_s2 = []
EPOCHS_S2 = 3

for e in range(EPOCHS_S2):
    train = train_one_epoch(efficientnet_b2_gpu, train_dataloader, optimizer_2, criterion, device)
    val = validate(efficientnet_b2_gpu, val_dataloader, criterion, device)

    if val < best_val_loss:
        best_val_loss = val
        torch.save(efficientnet_b2_gpu.state_dict(), model_path)

    print(f"EPOCH [{e + 1}/{EPOCHS_S2}] : train = {train} | val = {val}")
    train_loss_s2.append(train)
    val_loss_s2.append(val)

In [ ]:
efficientnet_b2_gpu.load_state_dict(torch.load(model_path))
submit(efficientnet_b2_gpu, test_dataloader, device)